# DictaLM LoRA fine-tuning for candidate selection (Colab, self-contained)

Trains LoRA adapters on a DictaLM causal language model for **candidate selection**,
using `data/study_v1/encoder_inputs/train.csv` and the 62-item natural dev set, the
same inputs as the DictaBERT cross-encoder. Each item becomes the exact selection
prompt the other LLM arms receive, and the target is the letter of the reference
candidate. Scoring reuses the shared selection evaluator, so candidate order, letter
parsing and accuracy are comparable with the Qwen and Gemini selection arms.

The notebook fetches the few source files it needs from GitHub, as
`run_test_eval_colab.ipynb` does; there is no clone and no package install.
It reads from the `main` branch.

**Runtime:** an A100 or L4 GPU trains a 7B model in bf16 (about 20–30 minutes for one
epoch). On a T4, 4-bit loading is selected automatically (about 1–1.5 hours).

The model ID and training settings are engineering choices for a first run, not an
approved protocol. Test-set evaluation is off (`RUN_TEST = False`) until it is
explicitly authorized.

## 1. Install and fetch source files

In [ ]:
# Colab preinstalls torchao 0.10, which this peft version rejects; training does not use it.
!pip uninstall -q -y torchao
!pip install -q transformers==4.57.6 peft==0.21.2 accelerate==1.15.0 bitsandbytes

In [ ]:
import urllib.request, sys, types, importlib.util

REPO_RAW = 'https://raw.githubusercontent.com/BenCarmel123/hebrew-acronym-disambiguation/main'

FILES = {
    'hebrew_acronyms.data_processing.common.csv_io': 'src/hebrew_acronyms/data_processing/common/csv_io.py',
    'hebrew_acronyms.models.common.pairs': 'src/hebrew_acronyms/models/common/pairs.py',
    'hebrew_acronyms.models.common.eval': 'src/hebrew_acronyms/models/common/eval.py',
    'hebrew_acronyms.models.dictalm_lora.examples': 'src/hebrew_acronyms/models/dictalm_lora/examples.py',
    'hebrew_acronyms.models.dictalm_lora.training': 'src/hebrew_acronyms/models/dictalm_lora/training.py',
    'hebrew_acronyms.models.dictalm_lora.eval': 'src/hebrew_acronyms/models/dictalm_lora/eval.py',
}

# Empty package stubs for every intermediate dotted path, so the modules' own
# `from hebrew_acronyms... import ...` lines resolve without an install.
for dotted in FILES:
    parts = dotted.split('.')
    for i in range(1, len(parts)):
        pkg = '.'.join(parts[:i])
        if pkg not in sys.modules:
            mod = types.ModuleType(pkg)
            mod.__path__ = []
            sys.modules[pkg] = mod

def fetch_module(dotted_name, repo_path):
    local_path = dotted_name.replace('.', '_') + '.py'
    urllib.request.urlretrieve(f'{REPO_RAW}/{repo_path}', local_path)
    spec = importlib.util.spec_from_file_location(dotted_name, local_path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[dotted_name] = mod
    spec.loader.exec_module(mod)
    return mod

# FILES is listed in dependency order.
for dotted, path in FILES.items():
    fetch_module(dotted, path)

from hebrew_acronyms.data_processing.common.csv_io import load_rows
from hebrew_acronyms.models.common.eval import write_details_csv
from hebrew_acronyms.models.dictalm_lora import training
from hebrew_acronyms.models.dictalm_lora.eval import evaluate_selection, load_adapter
print('modules loaded')

In [ ]:
for name in ('data/study_v1/encoder_inputs/train.csv', 'data/study_v1/encoder_inputs/dev.csv',
             'data/splits/test_items.csv'):
    urllib.request.urlretrieve(f'{REPO_RAW}/{name}', name.rsplit('/', 1)[-1])
train_rows, dev_rows, test_rows = load_rows('train.csv'), load_rows('dev.csv'), load_rows('test_items.csv')
print('train', len(train_rows), '| dev', len(dev_rows), '| test', len(test_rows))

## 2. Settings

In [ ]:
import torch

MODEL_ID = 'dicta-il/dictalm2.0-instruct'   # researcher's choice; check the model card
REVISION = None                             # set a commit SHA to pin the weights
BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
DTYPE = 'bfloat16' if BF16 else 'float16'
LOAD_IN_4BIT = not BF16                     # T4: 4-bit weights to fit in 16 GB
RUN_TEST = False                            # test evaluation needs explicit authorization
OUTPUT_DIR = 'dictalm-lora-run'

CONFIG = training.LoraTrainingConfig()      # engineering defaults; printed below
print(CONFIG)
print('dtype', DTYPE, '| 4-bit', LOAD_IN_4BIT)

## 3. Load the model

In [ ]:
tok, model = training.load_base_model(MODEL_ID, REVISION, dtype=DTYPE, load_in_4bit=LOAD_IN_4BIT)

## 4. Untrained test accuracy (if authorized)

In [ ]:
if RUN_TEST:
    before_test = evaluate_selection(test_rows, model, tok)
    print('untrained test accuracy', before_test['accuracy'], '| invalid', before_test['invalid_rate'])
    write_details_csv('test_untrained_details.csv', before_test['details'])

## 5. Train

The development loss after each epoch selects the saved adapter; it is not reported
as an accuracy.

In [ ]:
model = training.add_lora(model, CONFIG, quantized=LOAD_IN_4BIT)
model.print_trainable_parameters()
run_info = {'model_id': MODEL_ID, 'revision': REVISION,
            'resolved_commit': getattr(model.config, '_commit_hash', None),
            'dtype': DTYPE, 'load_in_4bit': LOAD_IN_4BIT, 'repo_raw': REPO_RAW}
history = training.train(model, tok, train_rows, dev_rows, OUTPUT_DIR, CONFIG, run_info)
history

## 6. Reload the selected adapter and evaluate on test (if authorized)

The base model is reloaded and the saved adapter applied, so evaluation uses the
checkpoint chosen by development loss, not the last training state.

In [ ]:
if RUN_TEST:
    import gc
    del model
    gc.collect()
    torch.cuda.empty_cache()
    tok, model = training.load_base_model(MODEL_ID, REVISION, dtype=DTYPE, load_in_4bit=LOAD_IN_4BIT)
    model = load_adapter(model, f'{OUTPUT_DIR}/adapter')
    after_test = evaluate_selection(test_rows, model, tok)
    print('fine-tuned test accuracy', after_test['accuracy'], '| invalid', after_test['invalid_rate'])
    write_details_csv('test_finetuned_details.csv', after_test['details'])

## 7. Download the adapter and per-item results

In [ ]:
!zip -q -r dictalm-lora-results.zip {OUTPUT_DIR} $(ls *_details.csv 2>/dev/null)
from google.colab import files
files.download('dictalm-lora-results.zip')